In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# ------------------------------------
# 1. 데이터 로드
# ------------------------------------
df = pd.read_csv(
    "/Users/mac/Desktop/project/company_data/third_week/data_csv/강원일반음식점.csv",
    encoding="CP949",
    low_memory=False
)

# ------------------------------------
# 2. 최근 5년 데이터 필터링
# ------------------------------------
df["인허가일자"] = pd.to_datetime(df["인허가일자"], errors="coerce")
df = df[df["인허가일자"] >= "2019-01-01"]

# ------------------------------------
# 3. 폐업 여부 및 결측치 처리
# ------------------------------------
df["폐업여부"] = df["폐업일자"].notnull().astype(int)

# ------------------------------------
# 4. 연도 컬럼 생성
# ------------------------------------
df["year"] = df["인허가일자"].dt.year

# ------------------------------------
# 5. 업태별 연도별 성장 지표 생성 (신규 - 폐업)
# ------------------------------------
annual = (
    df.groupby(["위생업태명", "year"])
      .agg(
          신규_count=("인허가일자", "count"),
          폐업_count=("폐업여부", "sum")
      )
      .reset_index()
)
annual["net_growth"] = annual["신규_count"] - annual["폐업_count"]

# ------------------------------------
# 6. 성장 지표 병합
# ------------------------------------
df = df.merge(
    annual[["위생업태명", "year", "net_growth"]],
    on=["위생업태명", "year"],
    how="left"
)

# ------------------------------------
# 7. 업태 더미 생성
# ------------------------------------
df = pd.get_dummies(df, columns=["위생업태명"], drop_first=True)

# ------------------------------------
# 8. 모델 입력 변수 설정 (소재지면적 제거)
# ------------------------------------
use_cols = ["net_growth"] + [c for c in df.columns if "위생업태명_" in c]
x = df[use_cols]
y = df["폐업여부"]

# ------------------------------------
# 9. 데이터 분리
# ------------------------------------
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

# ------------------------------------
# 10. 로지스틱 회귀 모델 학습
# ------------------------------------
model = LogisticRegression(max_iter=2000)
model.fit(x_train, y_train)

# ------------------------------------
# 11. 계수 정리
# ------------------------------------
coeff = (
    pd.DataFrame({"feature": x_train.columns, "coef": model.coef_[0]})
      .sort_values("coef", ascending=False)
)

print(coeff)

                  feature      coef
2                위생업태명_기타  0.871677
21               위생업태명_한식  0.640225
16             위생업태명_키즈카페  0.622367
19          위생업태명_패밀리레스트랑  0.413617
11  위생업태명_외국음식전문점(인도,태국등)  0.386956
4                위생업태명_까페  0.386098
8                위생업태명_분식  0.382868
23               위생업태명_횟집  0.196853
13       위생업태명_정종/대포집/소주방  0.077659
22            위생업태명_호프/통닭  0.070151
6             위생업태명_라이브카페  0.067984
3           위생업태명_김밥(도시락)  0.067363
0              net_growth -0.000477
1               위생업태명_경양식 -0.077753
9               위생업태명_뷔페식 -0.083058
7              위생업태명_복어취급 -0.166859
20            위생업태명_패스트푸드 -0.200807
17          위생업태명_탕류(보신용) -0.307478
18           위생업태명_통닭(치킨) -0.383173
14              위생업태명_중국식 -0.416078
10         위생업태명_식육(숯불구이) -0.422241
5               위생업태명_냉면집 -0.439357
12               위생업태명_일식 -0.441392
15             위생업태명_출장조리 -0.840192


In [3]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

# 예측값
y_pred = model.predict(x_test)
y_prob = model.predict_proba(x_test)[:, 1]

# 신뢰도 지표 계산
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc = roc_auc_score(y_test, y_prob)

print("Accuracy :", acc)
print("Precision:", prec)
print("Recall   :", rec)
print("F1 Score :", f1)
print("ROC-AUC  :", roc)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:\n", cm)

# 상세 리포트
print("\nClassfication Report:\n", classification_report(y_test, y_pred, zero_division=0))


Accuracy : 0.5850654349499615
Precision: 0.4824561403508772
Recall   : 0.034097954122752634
F1 Score : 0.06369426751592357
ROC-AUC  : 0.5793096372186145

Confusion Matrix:
 [[2225   59]
 [1558   55]]

Classfication Report:
               precision    recall  f1-score   support

           0       0.59      0.97      0.73      2284
           1       0.48      0.03      0.06      1613

    accuracy                           0.59      3897
   macro avg       0.54      0.50      0.40      3897
weighted avg       0.54      0.59      0.46      3897

